In [ ]:
!pip install SPARQLWrapper
!pip install sparql-dataframe

In [ ]:
import sparql_dataframe

endpoint = "http://localhost:7200/repositories/mqtt4ssn"

In [ ]:
NS = {
    "mqtt:": "https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#",
    "rdfs:": "http://www.w3.org/2000/01/rdf-schema#",
    "ex:": "http://example.org/",
    "sosa": "http://www.w3.org/ns/sosa/" 
}

In [ ]:
import pandas as pd

def strip_namespaces(df, ns_map=NS):
    def shrink(val):
        if not isinstance(val, str):
            return val
        # ersetze bekannte Namespaces durch Prefix
        for pref, base in ns_map.items():
            if val.startswith(base):
                return pref + val[len(base):]
        # fallback: nur local name nach # oder /
        if "#" in val:
            return val.rsplit("#", 1)[-1]
        if "/" in val:
            return val.rsplit("/", 1)[-1]
        return val

    return df.map(shrink)


## Fixed and Variable Headers

### CQ2.01u - Which MQTT PUBLISH packet has which MQTT Topic (name) in the Variable Header?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet ?topic WHERE {
  ?packet a mqtt:PublishPacket ;
          mqtt:hasPublishVariableHeader ?variable .
  ?variable mqtt:hasTopicName ?topic .
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ2.02u - Which QoS levels are set (in PUBLISH Fixed Header or SubscriptionEntry)?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?entity ?qos WHERE {
  { ?entity a mqtt:PublishFixedHeader ; 
        mqtt:hasQoSLevel ?qos . }
  UNION
  { ?entity a mqtt:SubscriptionEntry ; 
        mqtt:hasQoSLevel ?qos . }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ2.03u - Are RETAIN/DUP flags set in PUBLISH?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet ?retain ?dup WHERE {
  ?packet a mqtt:PublishPacket ;
          mqtt:hasPublishFixedHeader ?header .
  OPTIONAL { ?header mqtt:hasRetainFlag ?retain }
  OPTIONAL { ?header mqtt:hasDUPFlag ?dup }
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short

### CQ2.04u - Which MQTT Control Packets have which PacketIdentifier?

In [ ]:
q = """

PREFIX mqtt: <https://doernern.github.io/MQTT4SSNOntology/MQTT4SSN.owl#>

SELECT ?packet  ?type ?id WHERE {
  ?packet a mqtt:ControlPacket ;
          a ?type ;
          mqtt:hasVariableHeader ?variable .
  ?variable mqtt:hasPacketIdentifier ?id .

  ?type rdfs:subClassOf* mqtt:ControlPacket .

  FILTER( STRSTARTS(STR(?type), STR(mqtt:)) )
  FILTER( ?type != mqtt:ControlPacket )
}

"""

df = sparql_dataframe.get(endpoint, q)
df_short = strip_namespaces(df)
df_short